# Evaluate a selected training run
Use Godfather. Select RUN below. Load its best classifier and corresponding frozen DeBERTa, then reuse or rebuild the correct validation features. No training occurs.


In [ ]:
from pathlib import Path
import sys
import json
import math
import torch
import pandas as pd
from torch.utils.data import DataLoader
from IPython.display import display

ROOT = Path(r'C:/Users/ESHAAN/BYTE/Applied-Task/Magi/FINETUNING-EXPERMENTS/NarrationModelwBubbles')
sys.path.insert(0, str(ROOT))
for name in ['config', 'model', 'bubble_features', 'checkpoint_inputs']:
    sys.modules.pop(name, None)
from config import CONFIG, TRAINING_PATHS, save_json
from model import BoxClassifier, collate, metrics, prediction_rows
from bubble_features import sha256, model_files
from checkpoint_inputs import prepare_encoder, frozen_samples

RUN = TRAINING_PATHS['classifier_run']
if not (RUN / 'best.pt').exists():
    RUN = TRAINING_PATHS['deberta_run']
CHECKPOINT_PATH = RUN / 'best.pt'
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'No best.pt in {RUN}; select a completed training run.')
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
print('Evaluating:', CHECKPOINT_PATH, '| selected epoch:', checkpoint['epoch'])


## Load the encoder paired with this classifier
Joint runs export their best encoder under RUN/deberta. Classifier-only runs record their selected frozen encoder in encoder_source. A joint best.pt can reconstruct its encoder if the export folder is unavailable.


In [ ]:
if (RUN / 'deberta/config.json').is_file():
    encoder_input = RUN / 'deberta'
    if model_files(encoder_input) != checkpoint['feature_contract']['deberta_files']:
        encoder_input = CHECKPOINT_PATH
elif 'deberta' in checkpoint.get('models', {}):
    encoder_input = CHECKPOINT_PATH
else:
    encoder_input = Path(checkpoint.get('encoder_source', CONFIG['deberta_model']))
ENCODER_PATH = prepare_encoder(Path(CONFIG['deberta_model']), encoder_input, {}, CONFIG['cache_dir'])
base = torch.load(CONFIG['cache_dir'] / 'samples.pt', map_location='cpu', weights_only=True)
if ENCODER_PATH.resolve() == Path(CONFIG['deberta_model']).resolve():
    data = base
else:
    data = frozen_samples(base, ENCODER_PATH, CONFIG['cache_dir'], CONFIG['max_tokens'])
assert checkpoint['samples_signature'] in {base['signature'], data['signature']}, 'Training inputs changed.'
print('Validation encoder:', ENCODER_PATH)
print('Validation sequences:', len(data['val_indices']))
PREDICTIONS = RUN / 'evaluation_predictions.json'
META = RUN / 'evaluation_meta.json'
identity = {'checkpoint_sha256': sha256(CHECKPOINT_PATH), 'samples_signature': data['signature']}
reuse = PREDICTIONS.exists() and META.exists() and json.loads(META.read_text(encoding='utf-8')) == identity


## Score validation and save predictions
Metrics exclude ignored labels. This uses the original held-out validation split without oversampling. Results are written into the selected RUN, with caches tied to both classifier and text-feature identity.


In [ ]:
if reuse:
    rows = json.loads(PREDICTIONS.read_text(encoding='utf-8'))
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    classifier = BoxClassifier(checkpoint['visual_dim'], checkpoint['max_positions'],
                               **checkpoint['model_args']).to(device)
    classifier.load_state_dict(checkpoint['state_dict'], strict=True)
    classifier.eval()
    val = [data['samples'][i] for i in data['val_indices']]
    loader = DataLoader(val, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate)
    rows = []
    with torch.inference_mode():
        for batch in loader:
            batch = {key: value.to(device) if torch.is_tensor(value) else value for key, value in batch.items()}
            rows.extend(prediction_rows(batch, classifier(batch).softmax(-1).cpu()))
    save_json(PREDICTIONS, rows)
    save_json(META, identity)
report = metrics(rows)
labeled = [row for row in rows if row['label'] != -100]
weights = checkpoint['class_weights'].tolist()
report['val_loss'] = sum(-weights[row['label']] * math.log(max(
    row['p_spoken'] if row['label'] == 0 else row['p_narrated'], 1e-30)) for row in labeled) / len(labeled)
save_json(RUN / 'evaluation_metrics.json', report)
frame = pd.DataFrame(rows)
frame.to_csv(RUN / 'validation_predictions.csv', index=False)
errors = frame[(frame.label != -100) & (frame.label != frame.prediction)]
errors.to_csv(RUN / 'validation_errors.csv', index=False)
print(json.dumps(report, indent=2))
display(pd.crosstab(frame.loc[frame.label != -100, 'gt'], frame.loc[frame.label != -100, 'predicted_class']))
display(errors[['sequence_id', 'box_id', 'ocr', 'gt', 'predicted_class', 'p_spoken', 'p_narrated']].head(30))
